In [1]:
# ============================================================
# PLAGIARISM DETECTOR
# AIDE / STUDENT ESSAY DATASET
# TF-IDF + COSINE SIMILARITY
# ============================================================

import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# STEP 1: LOAD AIDE DATASET
# ============================================================

# CHANGE THIS if your downloaded CSV has a different name
file_path = "aide_dataset.csv"

df = pd.read_csv(
    file_path,
    encoding="utf-8",
    on_bad_lines="skip"
)

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())


# ============================================================
# STEP 2: FIND THE ESSAY/TEXT COLUMN
# ============================================================

# Possible column names containing essay text
possible_columns = [
    "essay",
    "text",
    "full_text",
    "essay_text",
    "document",
    "content"
]

text_column = None

for column in possible_columns:

    if column in df.columns:
        text_column = column
        break


if text_column is None:

    print("\nEssay text column was not automatically found.")
    print("Available columns:")
    print(df.columns.tolist())

    raise ValueError(
        "Please set the correct essay text column."
    )


print("\nText column selected:")
print(text_column)


# ============================================================
# STEP 3: REMOVE MISSING ESSAYS
# ============================================================

df = df.dropna(
    subset=[text_column]
)

# Convert essay text to string
df[text_column] = df[text_column].astype(str)

print("\nDataset shape after removing missing essays:")
print(df.shape)


# ============================================================
# STEP 4: REMOVE DUPLICATE ESSAYS
# ============================================================

duplicate_count = df[text_column].duplicated().sum()

print("\nDuplicate essays:")
print(duplicate_count)

df = df.drop_duplicates(
    subset=[text_column]
)

print("\nShape after removing duplicates:")
print(df.shape)


# ============================================================
# STEP 5: CLEAN AND NORMALIZE TEXT
# ============================================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation and special characters
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


df["cleaned_text"] = df[text_column].apply(
    clean_text
)

print("\nText cleaning completed.")


# ============================================================
# STEP 6: TOKENIZATION
# ============================================================

df["tokens"] = df["cleaned_text"].apply(
    lambda x: x.split()
)

print("Tokenization completed.")


# Display first essay tokens
print("\nFirst Essay Tokens:")
print(df["tokens"].iloc[0][:30])


# ============================================================
# STEP 7: SELECT DOCUMENTS
# ============================================================

# To keep the program fast, compare the first 100 essays.
# You can increase this number if your computer is fast.

number_of_documents = min(
    100,
    len(df)
)

documents = df["cleaned_text"].iloc[
    :number_of_documents
].tolist()

document_names = [
    f"Essay_{i+1}"
    for i in range(number_of_documents)
]


print("\nDocuments used for comparison:")
print(number_of_documents)


# ============================================================
# STEP 8: TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(
    documents
)

print("\nTF-IDF conversion completed.")

print(
    "TF-IDF matrix shape:",
    tfidf_matrix.shape
)


# ============================================================
# STEP 9: COSINE SIMILARITY
# ============================================================

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("\nCosine similarity calculated.")


# ============================================================
# STEP 10: SET PLAGIARISM THRESHOLD
# ============================================================

# 50% similarity
threshold = 0.50


# ============================================================
# STEP 11: COMPARE ALL DOCUMENT PAIRS
# ============================================================

results = []

for i in range(
    number_of_documents
):

    for j in range(
        i + 1,
        number_of_documents
    ):

        similarity = similarity_matrix[i][j]

        similarity_percentage = (
            similarity * 100
        )

        if similarity >= threshold:

            status = "Potential Plagiarism"

        else:

            status = "Low Similarity"


        results.append({

            "Document 1":
                document_names[i],

            "Document 2":
                document_names[j],

            "Similarity (%)":
                round(
                    similarity_percentage,
                    2
                ),

            "Status":
                status
        })


# ============================================================
# STEP 12: CREATE SIMILARITY REPORT
# ============================================================

report = pd.DataFrame(
    results
)

report = report.sort_values(
    by="Similarity (%)",
    ascending=False
)

report = report.reset_index(
    drop=True
)


print("\n")
print("=" * 75)
print("PLAGIARISM SIMILARITY REPORT")
print("=" * 75)

print(
    report.head(20).to_string(
        index=False
    )
)


# ============================================================
# STEP 13: DISPLAY POTENTIAL PLAGIARISM
# ============================================================

plagiarism_cases = report[
    report["Similarity (%)"] >=
    threshold * 100
]


print("\n")
print("=" * 75)
print("POTENTIAL PLAGIARISM CASES")
print("=" * 75)


if len(plagiarism_cases) > 0:

    print(
        plagiarism_cases.to_string(
            index=False
        )
    )

else:

    print(
        "No document pairs exceeded "
        f"the {threshold * 100}% threshold."
    )


# ============================================================
# STEP 14: MOST SIMILAR DOCUMENT PAIRS
# ============================================================

print("\n")
print("=" * 75)
print("TOP 10 MOST SIMILAR DOCUMENT PAIRS")
print("=" * 75)

print(
    report.head(10).to_string(
        index=False
    )
)


# ============================================================
# STEP 15: EXPORT REPORT
# ============================================================

output_file = (
    "Plagiarism_Similarity_Report.csv"
)

report.to_csv(
    output_file,
    index=False
)


# ============================================================
# STEP 16: FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 75)
print("PLAGIARISM DETECTION COMPLETED SUCCESSFULLY!")
print("=" * 75)

print(
    "\nOutput file:",
    output_file
)

FileNotFoundError: [Errno 2] No such file or directory: 'aide_dataset.csv'

In [1]:
import os

print("Current Jupyter folder:")
print(os.getcwd())

print("\nFiles available in this folder:")

for file in os.listdir():
    print(file)

Current Jupyter folder:
C:\Users\lenovo\anaconda3\NLPSkill

Files available in this folder:
.ipynb_checkpoints
28.7.ipynb
AIDE_train_essays.csv
archive (2).zip
ClassTask
Customer_Review_Cleaning_System.ipynb
IMDB Dataset.csv
IMDB_Dataset_CLEANED.csv
IMDB_Preprocessed_Dataset.csv
Instructions to recreate AI Generated Text Dataset.docx
missp.dat.txt
NLPtask(6.7).ipynb
plagiarism_corpus
Search_Query_Spelling_Corrector.ipynb
Smart_Next_Word_Predictor.ipynb
test.txt
train.txt
train_prompts.csv
Untitled.ipynb


In [2]:
# ============================================================
# PLAGIARISM DETECTOR USING AIDE TRAIN ESSAYS DATASET
# TF-IDF + COSINE SIMILARITY
# ============================================================

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# STEP 1: LOAD DATASET
# ============================================================

file_path = "AIDE_train_essays.csv"

df = pd.read_csv(
    file_path,
    encoding="utf-8",
    on_bad_lines="skip"
)

print("Dataset Loaded Successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())


# ============================================================
# STEP 2: DISPLAY FIRST 5 ROWS
# ============================================================

print("\nFirst 5 Rows:")
print(df.head())


# ============================================================
# STEP 3: FIND TEXT COLUMN
# ============================================================

# Automatically search for the essay/text column

possible_columns = [
    "text",
    "essay",
    "full_text",
    "essay_text",
    "content",
    "generated_text"
]

text_column = None

for column in possible_columns:

    if column in df.columns:

        text_column = column
        break


if text_column is None:

    print("\nText column was not automatically found.")

    print("Available columns:")
    print(df.columns.tolist())

    raise ValueError(
        "Please check the column names shown above."
    )


print("\nText Column Used:")
print(text_column)


# ============================================================
# STEP 4: CHECK MISSING VALUES
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())


# ============================================================
# STEP 5: REMOVE MISSING TEXT
# ============================================================

df = df.dropna(
    subset=[text_column]
)

df[text_column] = df[text_column].astype(str)


print("\nShape after removing missing text:")
print(df.shape)


# ============================================================
# STEP 6: CHECK DUPLICATES
# ============================================================

duplicate_count = df[text_column].duplicated().sum()

print("\nDuplicate Essays:")
print(duplicate_count)


# Remove duplicate essays

df = df.drop_duplicates(
    subset=[text_column]
)

print("\nShape after removing duplicates:")
print(df.shape)


# ============================================================
# STEP 7: TEXT CLEANING
# ============================================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation and special characters
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


df["cleaned_text"] = df[text_column].apply(
    clean_text
)

print("\nText Cleaning Completed!")


# ============================================================
# STEP 8: TOKENIZATION
# ============================================================

df["tokens"] = df["cleaned_text"].apply(
    lambda x: x.split()
)

print("Tokenization Completed!")


print("\nFirst Essay Tokens:")
print(
    df["tokens"].iloc[0][:30]
)


# ============================================================
# STEP 9: SELECT DOCUMENTS
# ============================================================

# Use first 100 essays for comparison
# This keeps the program fast.

number_of_documents = min(
    100,
    len(df)
)

documents = df[
    "cleaned_text"
].iloc[
    :number_of_documents
].tolist()


document_names = [
    f"Essay_{i+1}"
    for i in range(
        number_of_documents
    )
]


print("\nNumber of Documents Compared:")
print(number_of_documents)


# ============================================================
# STEP 10: TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(
    documents
)

print("\nTF-IDF Conversion Completed!")

print(
    "TF-IDF Matrix Shape:",
    tfidf_matrix.shape
)


# ============================================================
# STEP 11: COSINE SIMILARITY
# ============================================================

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("\nCosine Similarity Calculation Completed!")


# ============================================================
# STEP 12: SET THRESHOLD
# ============================================================

# 50% similarity threshold

threshold = 0.50


# ============================================================
# STEP 13: COMPARE DOCUMENTS
# ============================================================

results = []

for i in range(
    number_of_documents
):

    for j in range(
        i + 1,
        number_of_documents
    ):

        similarity = similarity_matrix[i][j]

        similarity_percentage = (
            similarity * 100
        )

        # Classification

        if similarity >= threshold:

            status = "Potential Plagiarism"

        else:

            status = "Low Similarity"


        results.append({

            "Document 1":
                document_names[i],

            "Document 2":
                document_names[j],

            "Similarity (%)":
                round(
                    similarity_percentage,
                    2
                ),

            "Status":
                status
        })


# ============================================================
# STEP 14: CREATE REPORT
# ============================================================

report = pd.DataFrame(
    results
)


# Sort by similarity

report = report.sort_values(
    by="Similarity (%)",
    ascending=False
)

report = report.reset_index(
    drop=True
)


# ============================================================
# STEP 15: DISPLAY REPORT
# ============================================================

print("\n")
print("=" * 75)
print("PLAGIARISM SIMILARITY REPORT")
print("=" * 75)

print(
    report.head(20).to_string(
        index=False
    )
)


# ============================================================
# STEP 16: DISPLAY POTENTIAL PLAGIARISM
# ============================================================

plagiarism_cases = report[
    report["Similarity (%)"]
    >= threshold * 100
]


print("\n")
print("=" * 75)
print("POTENTIAL PLAGIARISM CASES")
print("=" * 75)


if len(plagiarism_cases) > 0:

    print(
        plagiarism_cases.to_string(
            index=False
        )
    )

else:

    print(
        "No documents exceeded the "
        f"{threshold * 100}% threshold."
    )


# ============================================================
# STEP 17: TOP 10 MOST SIMILAR PAIRS
# ============================================================

print("\n")
print("=" * 75)
print("TOP 10 MOST SIMILAR DOCUMENT PAIRS")
print("=" * 75)

print(
    report.head(10).to_string(
        index=False
    )
)


# ============================================================
# STEP 18: EXPORT REPORT
# ============================================================

output_file = (
    "Plagiarism_Similarity_Report.csv"
)

report.to_csv(
    output_file,
    index=False
)


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n")
print("=" * 75)
print("PLAGIARISM DETECTION COMPLETED SUCCESSFULLY!")
print("=" * 75)

print(
    "\nSimilarity report saved as:"
)

print(output_file)

Dataset Loaded Successfully!

Dataset Shape:
(1378, 4)

Column Names:
['id', 'prompt_id', 'text', 'generated']

First 5 Rows:
         id  prompt_id                                               text  \
0  0059830c          0  Cars. Cars have been around since they became ...   
1  005db917          0  Transportation is a large necessity in most co...   
2  008f63e3          0  "America's love affair with it's vehicles seem...   
3  00940276          0  How often do you ride in a car? Do you drive a...   
4  00c39458          0  Cars are a wonderful thing. They are perhaps o...   

   generated  
0          0  
1          0  
2          0  
3          0  
4          0  

Text Column Used:
text

Missing Values:
id           0
prompt_id    0
text         0
generated    0
dtype: int64

Shape after removing missing text:
(1378, 4)

Duplicate Essays:
0

Shape after removing duplicates:
(1378, 4)

Text Cleaning Completed!
Tokenization Completed!

First Essay Tokens:
['cars', 'cars', 'have', 